# 🤖 AI Engineering Fundamentals — Lezione 4
## Notebook Gruppo C

**ITS Novitas 4.0 | Giovedì 28/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "C"
MEMBRI = ["", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# ⚠️ Prima esecuzione: ChromaDB scarica Sentence Transformers (~90MB)
# Le dipendenze (anthropic, chromadb, ...) sono nel requirements.txt
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic, os, chromadb
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente
chroma_client = chromadb.Client()

DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

# Collection base per gli esercizi
def chunka_testo(testo, chunk_size=400, overlap=50):
    chunks = []
    start = 0
    while start < len(testo):
        chunk = testo[start:start+chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunka_testo(DOCUMENTO_WIDATA)
collection = chroma_client.get_or_create_collection("widata_c")
collection.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])

def chiedi_claude(messaggio, system=None, max_tokens=600):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

def cerca(domanda, n=3):
    risultati = collection.query(query_texts=[domanda], n_results=n)
    return risultati["documents"][0]

print(f"✅ Setup completato! {collection.count()} chunk indicizzati.")

---
## 🎯 Tema del Gruppo C: Quando RAG Fallisce

Esplorate i 4 casi di fallimento del RAG, imparate a diagnosticarli
e implementate i fix per ognuno.

---
### Esercizio 1 — Diagnosticare il retrieval *(guidato)*

Il primo passo del debug: stampare sempre i chunk recuperati.
Verificate se sono rilevanti per la domanda.

In [ ]:
# Esercizio 1 — tool di debug del retrieval

def debug_retrieval(domanda, n=3):
    """Stampa i chunk recuperati con score di rilevanza."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n,
        include=["documents", "distances"]
    )
    chunks_trovati = risultati["documents"][0]
    distanze = risultati["distances"][0]

    print(f"❓ Domanda: {domanda}")
    print(f"📄 Chunk recuperati: {len(chunks_trovati)}\n")

    for i, (chunk, dist) in enumerate(zip(chunks_trovati, distanze)):
        rilevanza = 1 - dist  # distanza coseno → similarità
        print(f"Chunk {i+1} (rilevanza: {rilevanza:.3f}):")
        print(f"{chunk[:200]}...")
        print()

    return chunks_trovati

# Testate con domande buone e cattive
domande = [
    "Qual è l'autonomia della batteria del sensore XS200?",  # buona
    "Come si chiama il CEO di WiData?",                      # non nel documento
    "sensore temperatura umidità",                           # vaga
    "Qual è il prezzo Enterprise?",                          # risposta assente
]

for domanda in domande:
    print("="*55)
    debug_retrieval(domanda)

# Osservazione: quando la rilevanza è alta il retrieval funziona bene?
# Esiste una soglia sotto cui il retrieval è inaffidabile?
# ...

---
### Esercizio 2 — Hallucination residua *(guidato)*

Confrontate il comportamento del chatbot con e senza
l'istruzione anti-hallucination nel system prompt.
Il modello inventa di più senza quell'istruzione?

In [ ]:
# Esercizio 2 — istruzione anti-hallucination

SYSTEM_SENZA = """
Sei l'assistente di WiData Srl.
Rispondi alle domande sui prodotti IoT.
"""

SYSTEM_CON = """
Sei l'assistente di WiData Srl.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì esattamente:
'Non ho questa informazione nei miei documenti.'
Non inventare mai dati, prezzi, specifiche o nomi.
"""

def chat_rag(domanda, system):
    chunks_trovati = cerca(domanda)
    contesto = "\n\n---\n\n".join(chunks_trovati)
    prompt = f"Documenti:\n\n{contesto}\n\n---\n\nDomanda: {domanda}"
    return chiedi_claude(prompt, system=system)

# Domande dove la risposta NON è nel documento
domande_trabocchetto = [
    "Qual è il prezzo del piano Enterprise?",
    "WiData ha sensori per ambienti subacquei?",
    "Chi è il fondatore di WiData?",
]

for domanda in domande_trabocchetto:
    print(f"\n{'='*55}")
    print(f"❓ {domanda}")

    # SENZA istruzione anti-hallucination
    print("\n🔴 SENZA istruzione anti-hallucination:")
    print(chat_rag(domanda, SYSTEM_SENZA))

    # CON istruzione anti-hallucination
    print("\n🟢 CON istruzione anti-hallucination:")
    print(chat_rag(domanda, SYSTEM_CON))

# Osservazione: senza l'istruzione esplicita il modello tende a "riempire i vuoti"
# inventando prezzi/nomi plausibili (hallucination), anche se non sono nei documenti.
# Con l'istruzione anti-hallucination risponde "Non ho questa informazione...":
# il system prompt non elimina del tutto il rischio, ma lo riduce moltissimo.

---
### Esercizio 3 — I 4 casi di fallimento *(libero)*

Riproducete intenzionalmente i 4 fallimenti RAG
e implementate il fix per ognuno.

In [ ]:
# Esercizio 3 — i 4 fallimenti con fix

def make_coll(nome, chunk_size, overlap):
    """Crea (o ricrea) una collection con i parametri di chunking dati."""
    try:
        chroma_client.delete_collection(nome)
    except Exception:
        pass
    ch = chunka_testo(DOCUMENTO_WIDATA, chunk_size=chunk_size, overlap=overlap)
    c = chroma_client.create_collection(nome)
    c.add(documents=ch, ids=[str(i) for i in range(len(ch))])
    return c

def cerca_coll(domanda, coll, n=3):
    return coll.query(query_texts=[domanda], n_results=n)["documents"][0]

domanda = "Qual è l'autonomia della batteria del sensore XS200?"

# ── CASO 1: Retrieval sbagliato ────────────────────────────────────
# Causa: chunk troppo piccoli non hanno contesto sufficiente
print("CASO 1: Retrieval sbagliato (chunk troppo piccoli)")
coll_piccola = make_coll("fail_small", chunk_size=50, overlap=0)
print("  chunk_size=50 →", cerca_coll(domanda, coll_piccola, n=1)[0])
# Fix: chunk_size più grande, che mantiene insieme l'informazione
coll_fix1 = make_coll("fix_size", chunk_size=400, overlap=50)
print("  FIX chunk_size=400 →", cerca_coll(domanda, coll_fix1, n=1)[0])
print()

# ── CASO 2: Chunk non contestuali ─────────────────────────────────
# Causa: nessun overlap — frasi spezzate a metà
print("CASO 2: Chunk non contestuali (overlap=0)")
coll_no_ov = make_coll("fail_noov", chunk_size=120, overlap=0)
print("  overlap=0 →", cerca_coll("impermeabilità e connettività", coll_no_ov, n=1)[0])
coll_fix2 = make_coll("fix_ov", chunk_size=120, overlap=50)
print("  FIX overlap=50 →", cerca_coll("impermeabilità e connettività", coll_fix2, n=1)[0])
print()

# ── CASO 3: Hallucination residua ─────────────────────────────────
print("CASO 3: Hallucination residua")
print("Fix: istruzione anti-hallucination nel system prompt (vedi Es. 2)")
print()

# ── CASO 4: Contesto troppo lungo ──────────────────────────────────
# Causa: n_results troppo alto — troppi chunk nel prompt
print("CASO 4: Contesto troppo lungo (n_results troppo alto)")
contesto_lungo = "\n\n".join(cerca(domanda, n=10))
contesto_corto = "\n\n".join(cerca(domanda, n=3))
print(f"  n=10 → {len(contesto_lungo)} caratteri di contesto")
print(f"  FIX n=3 → {len(contesto_corto)} caratteri di contesto")
print("  Meno chunk = prompt più corto, più economico e con meno rumore irrilevante.")

---
### Esercizio 4 — Sistema di valutazione automatica *(libero)*

Costruite un sistema che testa automaticamente la qualità del RAG
su un dataset di domande con risposte attese.
Per ogni risposta verificate se contiene la parola chiave attesa.

In [ ]:
# Esercizio 4 — valutazione automatica del RAG

SYSTEM_WIDATA = """
Sei l'assistente di WiData Srl.
Rispondi SOLO basandoti sui documenti forniti.
Se la risposta non è nei documenti, dì 'Non ho questa informazione.'
"""

dataset_valutazione = [
    {"domanda": "Qual è l'autonomia del sensore XS200?",       "atteso": "2 anni",    "nel_doc": True},
    {"domanda": "Quanti sensori gestisce GW500?",               "atteso": "1000",      "nel_doc": True},
    {"domanda": "Quanto costa il piano Pro?",                   "atteso": "49",        "nel_doc": True},
    {"domanda": "Qual è la classificazione IP del sensore?",    "atteso": "IP67",      "nel_doc": True},
    {"domanda": "WiData ha una sede a Milano?",                 "atteso": "non ho",    "nel_doc": False},
    {"domanda": "Qual è il prezzo Enterprise?",                 "atteso": "non ho",    "nel_doc": False},
]

def valuta_rag(dataset, n_chunks=3):
    """Valuta la qualità del RAG su un dataset."""
    corrette = 0
    risultati = []

    for caso in dataset:
        chunks_trovati = cerca(caso["domanda"], n=n_chunks)
        contesto = "\n\n---\n\n".join(chunks_trovati)
        prompt = f"Documenti:\n\n{contesto}\n\n---\n\nDomanda: {caso['domanda']}"
        risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)

        # Trovato se la parola/frase attesa compare nella risposta (case insensitive)
        trovato = caso["atteso"].lower() in risposta.lower()
        if trovato:
            corrette += 1
        risultati.append({"caso": caso, "risposta": risposta, "trovato": trovato})

    return corrette, risultati

# Eseguiamo la valutazione e stampiamo il report
corrette, risultati = valuta_rag(dataset_valutazione)

for r in risultati:
    icona = "✅" if r["trovato"] else "❌"
    caso = r["caso"]
    print(f"{icona} Domanda: {caso['domanda']} → Atteso: '{caso['atteso']}' → Trovato: {'Sì' if r['trovato'] else 'No'}")

print(f"\nScore finale: {corrette}/{len(dataset_valutazione)}")

# Conclusione:
# Le domande con risposta nel documento dovrebbero passare quasi tutte; i casi "fuori
# documento" passano se il modello risponde davvero "non ho questa informazione".
# Dove fallisce: tipicamente quando il retrieval non porta il chunk giusto (query vaga)
# o quando il modello inventa invece di ammettere di non sapere.
# Per migliorare: chunking/overlap ottimali (Gruppo B), istruzione anti-hallucination,
# n_results adeguato e, se serve, una soglia minima di rilevanza sul retrieval.

---
## 📊 Preparate la presentazione (5 slide)

1. **Come diagnosticare RAG** — stampare i chunk e la rilevanza
2. **Hallucination residua** — con vs senza istruzione anti-hallucination
3. **I 4 fallimenti** — causa e fix per ognuno
4. **Il vostro score di valutazione** — X/6 e cosa ha fallito
5. **La checklist del buon RAG** — le 5 cose da verificare sempre

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*